<a href="https://colab.research.google.com/github/chitta-behera/Machine-Learning/blob/master/spam_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
emails = [
    "Congratulations! You won a free lottery",
    "Claim your free prize now",
    "Limited time offer buy today",
    "Win cash now",
    "Meeting scheduled tomorrow",
    "Please send the project report",
    "Let's discuss the architecture",
    "Can we meet tomorrow"
]

labels = [
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0
]

In [3]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-z\s]', '', text)

    return text

In [4]:
emails = [clean_text(email) for email in emails]

print(emails)

['congratulations you won a free lottery', 'claim your free prize now', 'limited time offer buy today', 'win cash now', 'meeting scheduled tomorrow', 'please send the project report', 'lets discuss the architecture', 'can we meet tomorrow']


In [5]:
tokenized = []

for email in emails:

    tokenized.append(email.split())

print(tokenized)

[['congratulations', 'you', 'won', 'a', 'free', 'lottery'], ['claim', 'your', 'free', 'prize', 'now'], ['limited', 'time', 'offer', 'buy', 'today'], ['win', 'cash', 'now'], ['meeting', 'scheduled', 'tomorrow'], ['please', 'send', 'the', 'project', 'report'], ['lets', 'discuss', 'the', 'architecture'], ['can', 'we', 'meet', 'tomorrow']]


In [6]:
vocab = {}

index = 1

for sentence in tokenized:

    for word in sentence:

        if word not in vocab:

            vocab[word] = index

            index += 1

print(vocab)

{'congratulations': 1, 'you': 2, 'won': 3, 'a': 4, 'free': 5, 'lottery': 6, 'claim': 7, 'your': 8, 'prize': 9, 'now': 10, 'limited': 11, 'time': 12, 'offer': 13, 'buy': 14, 'today': 15, 'win': 16, 'cash': 17, 'meeting': 18, 'scheduled': 19, 'tomorrow': 20, 'please': 21, 'send': 22, 'the': 23, 'project': 24, 'report': 25, 'lets': 26, 'discuss': 27, 'architecture': 28, 'can': 29, 'we': 30, 'meet': 31}


In [7]:
encoded = []

for sentence in tokenized:

    temp = []

    for word in sentence:

        temp.append(vocab[word])

    encoded.append(temp)

print(encoded)

[[1, 2, 3, 4, 5, 6], [7, 8, 5, 9, 10], [11, 12, 13, 14, 15], [16, 17, 10], [18, 19, 20], [21, 22, 23, 24, 25], [26, 27, 23, 28], [29, 30, 31, 20]]


In [8]:
max_len = max(len(sentence) for sentence in encoded)

print(max_len)

6


In [9]:
padded = []

for sentence in encoded:

    while len(sentence) < max_len:

        sentence.append(0)

    padded.append(sentence)

print(padded)

[[1, 2, 3, 4, 5, 6], [7, 8, 5, 9, 10, 0], [11, 12, 13, 14, 15, 0], [16, 17, 10, 0, 0, 0], [18, 19, 20, 0, 0, 0], [21, 22, 23, 24, 25, 0], [26, 27, 23, 28, 0, 0], [29, 30, 31, 20, 0, 0]]


In [10]:
X = torch.LongTensor(padded)

y = torch.FloatTensor(labels).view(-1,1)

In [11]:
class EmailDataset(Dataset):

    def __init__(self,X,y):

        self.X = X

        self.y = y

    def __len__(self):

        return len(self.X)

    def __getitem__(self,index):

        return self.X[index], self.y[index]

In [12]:
dataset = EmailDataset(X,y)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

In [13]:
class SpamModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=len(vocab)+1,
            embedding_dim=8
        )

        self.fc = nn.Sequential(

            nn.Linear(8,16),

            nn.ReLU(),

            nn.Linear(16,8),

            nn.ReLU(),

            nn.Linear(8,1)
        )

    def forward(self,x):

        embedded = self.embedding(x)

        embedded = embedded.mean(dim=1)

        output = self.fc(embedded)

        return output

In [14]:
model = SpamModel()

In [15]:
criterion = nn.BCEWithLogitsLoss()

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [17]:
epochs = 50

for epoch in range(epochs):

    for X_batch, y_batch in loader:

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch} Loss {loss.item():.4f}"
        )

Epoch 0 Loss 0.6972
Epoch 10 Loss 0.1383
Epoch 20 Loss 0.0002
Epoch 30 Loss 0.0003
Epoch 40 Loss 0.0001


In [28]:
test_email = "Congrats for anniversary" #"free cash prize"

test_email = clean_text(test_email)

words = test_email.split()

encoded = []

for word in words:

    encoded.append(vocab.get(word,0))

In [29]:
while len(encoded) < max_len:

    encoded.append(0)

In [30]:
test = torch.LongTensor([encoded])

In [31]:
model.eval()

with torch.no_grad():

    output = model(test)

    probability = torch.sigmoid(output)

    print(probability)

    if probability.item() > 0.5:

        print("Spam")

    else:

        print("Not Spam")

tensor([[0.1270]])
Not Spam
